[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C38_Frameworks_Accel_Course/03_jax_xla/03_jax_xla.ipynb)

# 03 · JAX/XLA 函数式与变换（numpy 复刻 grad / vmap / pytree）

目标：复刻 JAX 的灵魂——**纯函数 + 可组合的函数变换**。用模块 01 的 autograd 思想实现 numpy 版 `grad`（函数→梯度函数）与 `vmap`（自动向量化），验证它们能**自由组合**；再从零写 **pytree** 的展平/装回/map。

路线：迷你 autograd → `grad(f)` 函数变换 → `vmap(f)` 自动批处理 → `grad`∘`vmap` 组合 → pytree → 函数式训练步 → ✏️ 练习（vmap 批处理 / grad 组合 / pytree 展平）→ 📖 答案 → 🧪 真实数据胶囊（对照 JAX）。

> 心智模型：**变换是「函数→函数」的高阶操作**；纯函数是它们能自由组合的前提。

## 1 · 迷你 autograd（为 grad 打底）

`grad` 底层是反向模式 AD（模块 01）。这里用一个紧凑的 `Box` 类（记录计算图 + VJP）打底，支持 `+ * @ sum tanh` 等，足够搭一个小网络。**这是为了把 `grad` 实现成函数变换**——不是重点，扫一眼即可。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def unbroadcast(g, shape):
    while g.ndim > len(shape): g = g.sum(axis=0)
    for ax, s in enumerate(shape):
        if s == 1 and g.shape[ax] != 1: g = g.sum(axis=ax, keepdims=True)
    return g.reshape(shape)

class Box:
    '''被追踪的张量：data + grad + _backward（VJP 闭包）。'''
    def __init__(self, data, _children=()):
        self.data = np.asarray(data, dtype=float)
        self.grad = np.zeros_like(self.data)
        self._prev = set(_children); self._backward = lambda: None
    @property
    def shape(self): return self.data.shape
    def __add__(self, o):
        o = o if isinstance(o, Box) else Box(o)
        out = Box(self.data + o.data, (self, o))
        def bw():
            self.grad = self.grad + unbroadcast(out.grad, self.shape)
            o.grad = o.grad + unbroadcast(out.grad, o.shape)
        out._backward = bw; return out
    def __mul__(self, o):
        o = o if isinstance(o, Box) else Box(o)
        out = Box(self.data * o.data, (self, o))
        def bw():
            self.grad = self.grad + unbroadcast(o.data * out.grad, self.shape)
            o.grad = o.grad + unbroadcast(self.data * out.grad, o.shape)
        out._backward = bw; return out
    def __matmul__(self, o):
        out = Box(self.data @ o.data, (self, o))
        def bw():
            self.grad = self.grad + out.grad @ o.data.T
            o.grad = o.grad + self.data.T @ out.grad
        out._backward = bw; return out
    def sum(self):
        out = Box(self.data.sum(), (self,))
        def bw(): self.grad = self.grad + np.ones_like(self.data) * out.grad
        out._backward = bw; return out
    def tanh(self):
        t = np.tanh(self.data); out = Box(t, (self,))
        def bw(): self.grad = self.grad + (1 - t*t) * out.grad
        out._backward = bw; return out
    __radd__ = __add__; __rmul__ = __mul__

def _backprop(root):
    topo, seen = [], set()
    def build(v):
        if v not in seen:
            seen.add(v)
            for ch in v._prev: build(ch)
            topo.append(v)
    build(root)
    root.grad = np.ones_like(root.data)
    for v in reversed(topo): v._backward()

x = Box(np.array([1.0, 2.0])); s = (x * x).sum()
_backprop(s)
assert np.allclose(x.grad, 2 * x.data)   # d sum(x^2)/dx = 2x
print('✅ 迷你 autograd 就绪（grad 的底座）')

## 2 · `grad`：函数 → 梯度函数

现在把上面的反向传播**包装成一个函数变换** `grad(f)`：输入纯函数 `f: ndarray -> 标量`，返回新函数 `x -> ∇f(x)`。
**对外只见 numpy 数组进、numpy 数组出**——这正是 JAX 的接口形态（无副作用、梯度是返回值）。

In [ ]:
def grad(f):
    '''函数变换：grad(f) 返回计算 f 梯度的新函数。底层 trace+反向，对外是纯函数。'''
    def grad_f(x):
        xb = Box(np.asarray(x, dtype=float))   # 包成可追踪的 Box
        out = f(xb)                            # 跑 f，建图
        assert out.data.ndim == 0, 'grad 要求 f 返回标量'
        _backprop(out)                         # 反向
        return xb.grad                         # 梯度作为【返回值】
    return grad_f

# f(x) = sum(tanh(x) ^ ... ) 这类
def f(x):
    return (x * x * 0.5).sum()                 # f = 0.5*||x||^2, grad = x

df = grad(f)                                  # <- 变换：得到一个新函数
x0 = rng.standard_normal(5)
g = df(x0)
print('grad(f)(x) =', np.round(g, 4))
print('期望  x    =', np.round(x0, 4))
assert np.allclose(g, x0, atol=1e-6)

# 与数值梯度对拍（保证 grad 正确）
def numgrad(fn, x, eps=1e-6):
    x = np.asarray(x, float); out = np.zeros_like(x)
    for i in range(x.size):
        xp = x.copy().ravel(); xm = x.copy().ravel()
        xp[i] += eps; xm[i] -= eps
        out.ravel()[i] = (fn(xp.reshape(x.shape)) - fn(xm.reshape(x.shape))) / (2*eps)
    return out
def f_plain(x): return float((x*x*0.5).sum())
assert np.allclose(df(x0), numgrad(f_plain, x0), atol=1e-5)
print('✅ grad 作为函数变换：grad(f)(x) 与数值梯度一致')

## 3 · `vmap`：自动向量化

`vmap(f)` 把写给**单样本**的 `f` 自动提升为**批量**版本：沿批轴应用并堆叠结果。
我们实现一个最简版（用沿批轴映射 + 堆叠表达语义；真实 JAX 会把每个原语整体向量化、更快），验证它与逐样本 for 循环**逐位一致**。

In [ ]:
def vmap(f, in_axes=0, out_axes=0):
    '''把单样本函数 f 提升为批量版本：沿 in_axes 取每个切片、应用 f、沿 out_axes 堆叠。'''
    def batched(X):
        X = np.asarray(X)
        B = X.shape[in_axes]
        slices = [np.take(X, i, axis=in_axes) for i in range(B)]
        outs = [np.asarray(f(s)) for s in slices]
        return np.stack(outs, axis=out_axes)
    return batched

# 单样本函数：f(x) = x · w  (点积, 标量)
w = rng.standard_normal(4)
def dot_w(x):
    return float(x @ w)

X = rng.standard_normal((6, 4))            # 一批 6 个样本
vf = vmap(dot_w)                           # <- 变换：批量版
batched_out = vf(X)
loop_out = np.array([dot_w(X[i]) for i in range(6)])   # 逐样本参考
assert np.allclose(batched_out, loop_out)
assert np.allclose(batched_out, X @ w)     # 也等于一次矩阵乘
print('vmap 输出形状', batched_out.shape, '== 逐样本循环 ✅')
print('✅ vmap：只写单样本逻辑，自动得到批量版（== for 循环，但表达成向量化）')

## 4 · 组合：`grad` ∘ `vmap` 自由叠加

函数变换的**威力在组合**。因为 `grad(f)`、`vmap(f)` 返回的还是纯函数，可以再被变换：
- `vmap(grad(f))`：对**每个样本各自**求梯度（per-sample grads）；
- `grad(批损失)`：对批损失求一个总梯度（普通训练）。

同样的 `f`，叠法不同，语义不同——验证两者都对。

In [ ]:
# 单样本损失：l(x) = 0.5 * sum(x^2)，单样本梯度 = x
def loss1(x):
    return (x * x * 0.5).sum()

X = rng.standard_normal((5, 3))

# (A) vmap(grad(loss1))：每个样本各自的梯度 -> 应等于 X 本身
per_sample_grad = vmap(grad(loss1))(X)
assert per_sample_grad.shape == (5, 3)
assert np.allclose(per_sample_grad, X, atol=1e-6)
print('vmap(grad(f)) -> per-sample 梯度, shape', per_sample_grad.shape, '✅')

# (B) grad(批损失): 对 sum over batch 求梯度 -> 也应是 X（每个元素 d/dx 0.5x^2 = x）
def batch_loss(Xb):
    return (Xb * Xb * 0.5).sum()
total_grad = grad(batch_loss)(X)
assert np.allclose(total_grad, X, atol=1e-6)
print('grad(批损失)   -> 总梯度,       shape', total_grad.shape, '✅')

# 关键：per-sample 梯度【求和】== 批损失梯度（因为 sum 可交换）
assert np.allclose(per_sample_grad, total_grad, atol=1e-6)
print('✅ 变换自由组合：vmap(grad) 与 grad(批) 的关系符合预期')

## 5 · pytree：穿过嵌套结构

真实模型参数是**嵌套的 dict/list**（每层一组 W/b）。pytree 把它拆成「扁平叶子列表 + 结构定义」，
让变换自动作用到每个叶子。从零写 `tree_flatten` / `tree_unflatten` / `tree_map`。

In [ ]:
def tree_flatten(tree):
    '''把嵌套 dict/list/tuple 拆成 (叶子列表, treedef)。叶子=非容器(np.ndarray/数)。'''
    if isinstance(tree, dict):
        keys = sorted(tree.keys())
        leaves, defs = [], []
        for k in keys:
            lv, d = tree_flatten(tree[k]); leaves += lv; defs.append((k, d))
        return leaves, ('dict', defs)
    if isinstance(tree, (list, tuple)):
        leaves, defs = [], []
        for v in tree:
            lv, d = tree_flatten(v); leaves += lv; defs.append(d)
        return leaves, ('list' if isinstance(tree, list) else 'tuple', defs)
    return [tree], ('leaf', None)        # 叶子

def tree_unflatten(treedef, leaves):
    '''用 treedef 把扁平叶子装回原结构。返回 (重建结构, 剩余叶子)。'''
    kind, spec = treedef
    if kind == 'leaf':
        return leaves[0], leaves[1:]
    if kind == 'dict':
        out = {}
        for k, d in spec:
            out[k], leaves = tree_unflatten(d, leaves)
        return out, leaves
    # list / tuple
    vals = []
    for d in spec:
        v, leaves = tree_unflatten(d, leaves); vals.append(v)
    return (vals if kind == 'list' else tuple(vals)), leaves

def tree_map(fn, *trees):
    '''对多个同构 pytree 的对应叶子应用 fn，返回同构 pytree。'''
    flats = [tree_flatten(t) for t in trees]
    leaves_list = [f[0] for f in flats]; treedef = flats[0][1]
    new_leaves = [fn(*ls) for ls in zip(*leaves_list)]
    out, rest = tree_unflatten(treedef, new_leaves)
    return out

params = {'layer1': {'W': np.ones((2,2)), 'b': np.zeros(2)},
          'layer2': {'W': np.ones((2,1)) * 3, 'b': np.zeros(1)}}
leaves, treedef = tree_flatten(params)
print('叶子数:', len(leaves), '(W1,b1,W2,b2)')
rebuilt, rest = tree_unflatten(treedef, leaves)
assert len(leaves) == 4 and len(rest) == 0
# 展平再装回 == 原结构
assert np.allclose(rebuilt['layer1']['W'], params['layer1']['W'])
assert np.allclose(rebuilt['layer2']['W'], params['layer2']['W'])
print('✅ tree_flatten/unflatten：展平再装回 == 原结构')

In [ ]:
# tree_map 演示：梯度下降更新（对每个叶子做 p -> p - lr*g）
grads = tree_map(lambda p: p * 0.1, params)        # 假装的梯度
lr = 0.5
new_params = tree_map(lambda p, g: p - lr * g, params, grads)
# 检查 layer1.W: 1 - 0.5*0.1 = 0.95
assert np.allclose(new_params['layer1']['W'], 0.95)
assert np.allclose(new_params['layer2']['W'], 3 - 0.5*0.3)
print('✅ tree_map：对所有叶子统一做梯度更新 —— 函数式处理任意复杂参数')

## 6 · 函数式训练步：无可变状态

把 grad + pytree 拼成一个**纯函数训练步**：`params, loss = step(params, X, y)`——参数进、参数出、无副作用。
这正是 JAX 训练循环的形态（整个 step 可被 jit/vmap/pmap）。验证 loss 单调下降。

In [ ]:
# 一个线性模型 y = X @ w + b，参数是 pytree {'w':..., 'b':...}
n, d = 40, 3
Xd = rng.standard_normal((n, d)); true_w = rng.standard_normal((d, 1))
yd = Xd @ true_w + 0.5                          # (n,1)
Xaug = np.concatenate([Xd, np.ones((n, 1))], axis=1)   # (n, d+1)，最后一列吸收 bias

def loss_box(theta):       # theta 是 Box, 形状 (d+1, 1)；全程 2-D 使 matmul VJP 干净
    pred = Box(Xaug) @ theta                    # (n,1)
    diff = pred + Box(yd) * -1                  # pred - y
    return (diff * diff).sum() * (1.0 / n)      # MSE（标量）

theta = np.zeros((d + 1, 1))
g_loss = grad(loss_box)
lr = 0.3; losses = []
for step in range(60):
    gr = g_loss(theta)                 # 纯函数：梯度是返回值（同形状 (d+1,1)）
    theta = theta - lr * gr            # 旧 theta -> 新 theta（无原地改）
    pred = Xaug @ theta
    losses.append(float(((pred - yd) ** 2).mean()))
print(f'初始 MSE={losses[0]:.4f}  最终 MSE={losses[-1]:.4f}')
assert losses[-1] < losses[0] * 0.1, 'loss 应大幅下降'
print('✅ 函数式训练步：参数进/出、无可变状态，grad 把 loss 训下去了')

---
## ✏️ 练习 1：用 vmap 批量化一个单样本函数

给一个单样本函数 `predict(x) = tanh(x @ W + b)`（输入 `(din,)`，输出 `(dout,)`），
用我们的 `vmap` 把它批量化，验证 `vmap(predict)(X)`（X 是 `(B, din)`）与逐样本循环一致。

实现 `batched_predict(X, W, b)`：返回 `vmap(单样本 predict)(X)`。

In [ ]:
Wp = rng.standard_normal((3, 2)); bp = rng.standard_normal(2)
def batched_predict(X, W, b):
    # TODO: 定义单样本 predict(x) = np.tanh(x @ W + b)，用 vmap 批量化后作用到 X，返回 (B, 2)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
Xb = rng.standard_normal((7, 3))
out = batched_predict(Xb, Wp, bp)
ref = np.stack([np.tanh(Xb[i] @ Wp + bp) for i in range(7)])
assert out.shape == (7, 2)
assert np.allclose(out, ref)
print('✅ 练习 1 通过：vmap 把单样本 predict 批量化，结果 == 逐样本循环')

## ✏️ 练习 2：grad 的组合——一个表达式的二阶导

JAX 的 `grad(grad(f))` 是二阶导。我们的标量 `grad` 暂不支持高阶（Box 不可二次微分），
所以这里换个**等价的组合**练习：用 `grad` + 数值微分**验证** `grad` 的输出本身是个可用的函数。

实现 `grad_then_sum(f, x)`：返回 `sum(grad(f)(x))`（梯度各分量之和），并验证它对 `f(x)=0.5*sum(x^2)` 等于 `sum(x)`。

In [ ]:
def grad_then_sum(f, x):
    # TODO: 用我们的 grad 得到梯度函数，对 x 求梯度，返回梯度所有分量之和(float)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
def fq(x): return (x * x * 0.5).sum()
xq = rng.standard_normal(6)
val = grad_then_sum(fq, xq)
assert np.isclose(val, xq.sum(), atol=1e-6), 'grad(0.5||x||^2)=x, 其和=sum(x)'
# grad(f) 确实是个可继续操作的函数（这里我们对它求和）
print(f'sum(grad(f)(x)) = {val:.4f}, sum(x) = {xq.sum():.4f}')
print('✅ 练习 2 通过：grad(f) 是返回值/普通函数，可继续组合操作')

## ✏️ 练习 3：pytree 叶子计数与 L2 范数

实现 `tree_l2(tree)`：把一个嵌套参数 pytree 的**所有叶子**展平，计算它们拼起来的全局 L2 范数 `sqrt(sum of all elements^2)`。用 `tree_flatten`。

In [ ]:
def tree_l2(tree):
    # TODO: tree_flatten 拿到叶子列表，对每个叶子算 sum(x^2)，加总后开方
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
tree = {'a': np.array([3.0, 4.0]), 'b': {'c': np.array([[0.0, 0.0]]), 'd': np.array([12.0])}}
# 全局: sqrt(3^2+4^2+0+0+12^2) = sqrt(9+16+144) = sqrt(169) = 13
val = tree_l2(tree)
assert np.isclose(val, 13.0), f'期望 13, 得到 {val}'
print(f'✅ 练习 3 通过：pytree 全局 L2 范数 = {val} (穿过任意嵌套)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def batched_predict(X, W, b):
    def predict(x):
        return np.tanh(x @ W + b)
    return vmap(predict)(X)

In [ ]:
# 练习 2 参考答案
def grad_then_sum(f, x):
    g = grad(f)(x)          # grad(f) 是函数；对 x 调用得梯度数组
    return float(np.sum(g))

In [ ]:
# 练习 3 参考答案
def tree_l2(tree):
    leaves, _ = tree_flatten(tree)
    total = sum(float((np.asarray(l) ** 2).sum()) for l in leaves)
    return float(np.sqrt(total))

---
## 🧪 真实数据胶囊：对照 JAX 的 grad/vmap

我们的 numpy `grad`/`vmap`，与真实 **JAX** 的 `jax.grad`/`jax.vmap` 做的是同一件事、给同样的结果。
下面对同一个函数对拍。

**装了 jax 才实跑对拍；没装则用我们的 numpy 版自洽对拍（不阻断）。** 本环境通常没有 jax，正好演示优雅回退。

In [ ]:
# 我们的 numpy 版：f(x)=sum(sin? ) 这里用 0.5||x||^2
def f_demo(x):
    return (x * x * 0.5).sum()
x_demo = rng.standard_normal(5)
ours_grad = grad(f_demo)(x_demo)
X_demo = rng.standard_normal((4, 5))
ours_vmap = vmap(grad(f_demo))(X_demo)
print('我们的 grad(f)(x) =', np.round(ours_grad, 4))
print('我们的 vmap(grad(f))(X) shape =', ours_vmap.shape)

**🧪 胶囊练习**：实现 `jax_or_numpy_check()`：
- 若有 jax：用 `jax.grad(f_demo)(x_demo)` 与 `jax.vmap(jax.grad(f_demo))(X_demo)`，返回它们是否与我们的 `ours_grad`/`ours_vmap` 一致；
- 若没 jax：直接验证我们的 `grad`/`vmap` 自洽（`grad` 对拍数值梯度、`vmap`==循环），返回 True。

学生骨架（不计入自动验证）：

In [ ]:
def jax_or_numpy_check():
    # TODO: 有 jax -> 对拍 jax.grad/jax.vmap；没 jax -> 自洽验证。返回 bool
    raise NotImplementedError

In [ ]:
# 自测（学生填好后运行）
assert jax_or_numpy_check() == True
print('✅ 胶囊通过：我们的 grad/vmap 与 JAX(或数值梯度)一致 —— 函数变换的语义对得上')

In [ ]:
# 📖 胶囊参考答案
def jax_or_numpy_check():
    try:
        import jax, jax.numpy as jnp
        jg = np.asarray(jax.grad(f_demo)(jnp.asarray(x_demo)))
        jv = np.asarray(jax.vmap(jax.grad(f_demo))(jnp.asarray(X_demo)))
        return bool(np.allclose(jg, ours_grad, atol=1e-5) and np.allclose(jv, ours_vmap, atol=1e-5))
    except Exception:
        # 回退：自洽验证（grad 对拍数值梯度、vmap==循环）
        def fp(x): return float((x*x*0.5).sum())
        ng = numgrad(fp, x_demo)
        loop = np.stack([grad(f_demo)(X_demo[i]) for i in range(4)])
        return bool(np.allclose(ours_grad, ng, atol=1e-5) and np.allclose(ours_vmap, loop, atol=1e-6))

---
## 🔧 旁注：真实 JAX 长什么样

我们手写的 `grad`/`vmap`/`tree_map`，在 JAX 里就是内建的可组合变换（对照，**不依赖即可读**）：

```python
import jax, jax.numpy as jnp

def loss(params, x, y):                 # 纯函数：参数显式传入
    pred = x @ params['w'] + params['b']
    return jnp.mean((pred - y) ** 2)

g = jax.grad(loss)(params, x, y)        # == 我们的 grad(loss)(params)；返回同构梯度 pytree
per_sample = jax.vmap(jax.grad(loss), in_axes=(None, 0, 0))(params, X, Y)  # grad∘vmap 组合
fast = jax.jit(loss)                    # XLA 编译版，首调编译、后续复用
new = jax.tree_util.tree_map(lambda p, g: p - 0.1*g, params, g)  # == 我们的 tree_map
print(jax.make_jaxpr(loss)(params, x, y))   # 看 IR（jaxpr）== 模块 02 的 IR
```

对应关系：我们的 `grad`/`vmap`/`tree_map` ↔ `jax.grad`/`jax.vmap`/`tree_map`；jaxpr ↔ 模块 02 的玩具 IR；jit ↔ torch.compile（都是 trace+编译+缓存）。

### 小结
- 两条路线：PyTorch **操作张量**（命令式、副作用），JAX **变换函数**（函数式、纯）。
- **纯函数**是变换可组合的前提：无副作用、状态/随机显式化、控制流用 lax 原语。
- `grad`/`vmap`/`jit` 都是 **(函数)→(函数)** 的高阶变换，可任意叠加（`jit(vmap(grad(f)))`）。
- `vmap` 只写单样本逻辑、自动批量化；`grad`∘`vmap` 给 per-sample 梯度。
- **pytree** 让变换穿过嵌套参数：`tree_flatten`/`unflatten`/`map`，处理复杂模型也只是「对叶子操作」。

下一站：**模块 04 · 混合精度与显存** —— 在固定硬件上把模型训得起、训得快而不 NaN/OOM。